# Chicago weather: analysis and practical plotting lessons

This notebook analyzes the existing weather CSV and teaches how to create and read
charts with **Matplotlib, Seaborn, and pandas plotting**. It covers distributions,
seasonality, rain versus snow, extremes, weather persistence, and hourly timing.
The final section explains how to join weather to the existing daily bus and rail totals.
No transportation data is read, changed, or merged by the executed cells.

### Run it in your repository

Save this file as `notebooks/01_weather_exploration.ipynb`. Keep the original input at
`datasets/open-meteo-41.86N87.65W179m.csv`. Open the notebook from the repository root
or `notebooks/` and choose **Kernel → Restart Kernel and Run All Cells**.
All code is included here; separate helper scripts are not required.

In a notebook, install dependencies once if needed:

```python
%pip install "pandas>=2.2,<3" "numpy>=1.26,<3" "matplotlib>=3.8,<4" "seaborn>=0.13,<0.14" scipy ipython
```

### Reading plan

Start with setup and data quality, then lessons 1–7 for plotting fundamentals. Lessons
8–19 introduce advanced chart layouts, transformations, uncertainty, and reshaping.
Each lesson gives the question, code pattern, interpretation, and practical cautions.
The exercises and merge plan are at the end.

**Units:** temperature in °C; precipitation and rain in mm; new snowfall in cm;
10 m sustained wind in km/h. Rain is part of total precipitation. New snowfall is
neither snow depth nor an additive amount of liquid precipitation.

In [1]:
from pathlib import Path
import calendar
import hashlib
import json
import os
import platform
import tempfile

os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir()) / 'chicago-weather-mpl'))
os.environ.setdefault('XDG_CACHE_HOME', str(Path(tempfile.gettempdir()) / 'chicago-weather-cache'))
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import seaborn as sns
from IPython.display import display, Image

ROOT = Path.cwd().resolve()
if not (ROOT / 'datasets').is_dir():
    ROOT = ROOT.parent
SOURCE = ROOT / 'datasets/open-meteo-41.86N87.65W179m.csv'
assert SOURCE.is_file(), 'Run from your repository root or notebooks/; keep the original CSV in datasets/.'
DATA = ROOT / 'datasets/processed/weather'
REPORT = ROOT / 'reports/weather'
FIGURES = ROOT / 'figures/weather/tutorial'
for folder in [DATA, REPORT, FIGURES]:
    folder.mkdir(parents=True, exist_ok=True)
TIMEZONE = 'America/Chicago'
BASELINE = (2011, 2025)
RENAME = {
    'temperature_2m (°C)': 'temperature_c',
    'precipitation (mm)': 'precipitation_mm',
    'rain (mm)': 'rain_mm',
    'snowfall (cm)': 'snowfall_cm',
    'wind_speed_10m (km/h)': 'wind_kmh',
}
MONTHS = [calendar.month_abbr[m] for m in range(1, 13)]
SEASONS = ['Winter', 'Spring', 'Summer', 'Autumn']
PALETTE = {'Rain only': '#1976a3', 'Snow only': '#7851a9',
           'Mixed': '#cc7c29', 'Neither threshold': '#bac4ce'}
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

def write_csv(frame, name, directory=DATA):
    frame.to_csv(directory / name, index=False, float_format='%.6f')

def show_figure(fig, filename):
    # Save a shareable PNG, display it in the notebook, then close the figure.
    path = FIGURES / filename
    fig.savefig(path, dpi=140, bbox_inches='tight')
    display(Image(filename=str(path)))
    plt.close(fig)

print('pandas', pd.__version__, '| Matplotlib', matplotlib.__version__, '| Seaborn', sns.__version__)
print('Input:', SOURCE.relative_to(ROOT))

AssertionError: Run from your repository root or notebooks/; keep the original CSV in datasets/.

## Load and validate the hourly source

The first CSV rows are metadata, followed by a blank line and the hourly header.
`skiprows=3` reads the actual weather table. GMT and a zero UTC offset establish that
the timestamps are UTC. We validate uniqueness, hourly continuity, finite values,
nonnegative amounts, and broad physical plausibility before aggregation.

**Why time zones matter:** convert UTC to `America/Chicago` before extracting the date.
Otherwise evening weather can be assigned to the next day in your ridership merge.
These checks establish internal consistency; they do not independently verify the
gridded weather estimates against Chicago stations.

In [ ]:
def load_hourly():
    metadata = pd.read_csv(SOURCE, nrows=1).iloc[0].to_dict()
    assert metadata["utc_offset_seconds"] == 0 and metadata["timezone"] == "GMT"
    raw = pd.read_csv(SOURCE, skiprows=3)
    assert list(raw.columns) == ["time", *RENAME]
    h = raw.rename(columns=RENAME)
    h["time_utc"] = pd.to_datetime(h.pop("time"), utc=True, errors="raise")
    assert not h["time_utc"].duplicated().any(), "Duplicate UTC timestamps"
    h = h.sort_values("time_utc").reset_index(drop=True)
    expected = pd.date_range(h.time_utc.min(), h.time_utc.max(), freq="h")
    assert h.time_utc.tolist() == expected.tolist(), "Unexpected hourly gap"
    assert not h[list(RENAME.values())].isna().any().any(), "Missing measurements"
    assert np.isfinite(h[list(RENAME.values())]).all().all(), "Non-finite value"
    assert (h[["precipitation_mm", "rain_mm", "snowfall_cm", "wind_kmh"]] >= 0).all().all()
    assert h.temperature_c.between(-60, 60).all(), "Investigate temperature range"
    assert (h.rain_mm <= h.precipitation_mm + 0.011).all(), "Rain exceeds precipitation"
    h["time_local"] = h.time_utc.dt.tz_convert(TIMEZONE)
    h["service_date"] = h.time_local.dt.tz_localize(None).dt.normalize()
    h["local_hour"] = h.time_local.dt.hour
    h["weekday"] = h.time_local.dt.dayofweek
    return h, metadata

source_checksum = hashlib.sha256(SOURCE.read_bytes()).hexdigest()
hourly, source_metadata = load_hourly()
display(pd.DataFrame([source_metadata]))
display(hourly.head())
display(hourly[list(RENAME.values())].describe().round(2))
print(f'{len(hourly):,} hourly records; no missing values, duplicate UTC timestamps, or hourly gaps.')
print('UTC coverage:', hourly.time_utc.min(), 'to', hourly.time_utc.max())
print('First Chicago timestamp:', hourly.time_local.iloc[0])

## Construct complete local days and weather features

`groupby(...).agg(...)` combines hourly measurements into daily mean/minimum/maximum
temperature, precipitation/rain/snow sums, mean/maximum wind, and exposure-hour counts.
The function also creates transparent event flags, selected weekday peak windows,
lagged features, and a descriptive monthly temperature anomaly.

Local days can contain **23, 24, or 25 hours**. We exclude incomplete boundary days;
we retain valid daylight-saving days and valid extreme measurements. No imputation,
outlier clipping, or scaling of partial-day precipitation totals is performed.

The anomaly reference is 2011–2025 and includes future data relative to earlier rows.
Use a training-only reference for prediction; the current anomaly is descriptive.
Peak-hour features use clock labels 07–09 and 16–18, Monday–Friday including holidays.
Weekend peak fields, the first lag value, and first three prior-three-day values
are intentionally missing.

In [ ]:
def aggregate_daily(h):
    g = h.groupby("service_date", sort=True)
    d = g.agg(
        observed_hours=("temperature_c", "size"),
        temperature_mean_c=("temperature_c", "mean"),
        temperature_min_c=("temperature_c", "min"),
        temperature_max_c=("temperature_c", "max"),
        precipitation_total_mm=("precipitation_mm", "sum"),
        precipitation_max_hourly_mm=("precipitation_mm", "max"),
        rain_total_mm=("rain_mm", "sum"),
        snowfall_total_cm=("snowfall_cm", "sum"),
        wind_mean_kmh=("wind_kmh", "mean"),
        wind_max_kmh=("wind_kmh", "max"),
    )
    for name, flag in {
        "wet_hours": h.precipitation_mm >= 0.1,
        "snow_hours": h.snowfall_cm > 0,
        "freezing_hours": h.temperature_c <= 0,
        "cold_hours_le_minus10c": h.temperature_c <= -10,
        "hot_hours_ge_30c": h.temperature_c >= 30,
        "windy_hours_ge_40kmh": h.wind_kmh >= 40,
    }.items():
        d[name] = flag.groupby(h.service_date).sum().astype(int)
    dates = d.index
    start = dates.tz_localize(TIMEZONE)
    end = (dates + pd.Timedelta(days=1)).tz_localize(TIMEZONE)
    d["expected_hours"] = ((end - start) / pd.Timedelta(hours=1)).astype(int)
    d["is_complete_day"] = d.observed_hours.eq(d.expected_hours)
    d["year"] = dates.year
    d["month"] = dates.month
    d["weekday"] = dates.day_name()
    d["is_weekend"] = dates.dayofweek >= 5
    d["season"] = pd.Series(dates.month, index=dates).map({
        12: "Winter", 1: "Winter", 2: "Winter", 3: "Spring", 4: "Spring", 5: "Spring",
        6: "Summer", 7: "Summer", 8: "Summer", 9: "Autumn", 10: "Autumn", 11: "Autumn",
    })
    # A DST transition day is complete with 23 or 25 hours, not necessarily 24.
    complete = d.loc[d.is_complete_day].copy()
    assert complete.index.equals(pd.date_range(complete.index.min(), complete.index.max(), freq="D", name="service_date"))
    for name, flag in {
        "wet_day": complete.precipitation_total_mm >= 1,
        "heavy_precipitation_day": complete.precipitation_total_mm >= 25,
        "snow_day": complete.snowfall_total_cm >= 0.1,
        "heavy_snow_day": complete.snowfall_total_cm >= 10,
        "cold_day": complete.temperature_min_c <= -10,
        "severe_cold_day": complete.temperature_min_c <= -20,
        "hot_day": complete.temperature_max_c >= 30,
        "very_hot_day": complete.temperature_max_c >= 35,
        "windy_day": complete.wind_max_kmh >= 40,
        "freeze_thaw_day": (complete.temperature_min_c < 0) & (complete.temperature_max_c > 0),
    }.items():
        complete[name] = flag
    adverse = ["heavy_precipitation_day", "heavy_snow_day", "cold_day", "hot_day", "windy_day"]
    complete["adverse_condition_count"] = complete[adverse].sum(axis=1)
    complete["any_adverse_condition"] = complete.adverse_condition_count > 0
    # Window endpoints are inclusive; all hours use local clock time.
    for label, hours in {"am_peak": [7, 8, 9], "pm_peak": [16, 17, 18]}.items():
        part = h.loc[(h.weekday < 5) & h.local_hour.isin(hours)].groupby("service_date")
        p = part.agg(temperature_mean_c=("temperature_c", "mean"),
                     precipitation_mm=("precipitation_mm", "sum"),
                     snowfall_cm=("snowfall_cm", "sum"), wind_max_kmh=("wind_kmh", "max"))
        complete = complete.join(p.add_prefix(label + "_"))
    for measure in ["precipitation_total_mm", "snowfall_total_cm", "temperature_min_c"]:
        complete[measure + "_lag1"] = complete[measure].shift(1)
    for measure in ["precipitation_total_mm", "snowfall_total_cm"]:
        complete[measure + "_prior3d"] = complete[measure].shift(1).rolling(3, min_periods=3).sum()
    baseline = complete[complete.year.between(*BASELINE)]
    monthly_normal = baseline.groupby("month").temperature_mean_c.mean()
    complete["temperature_month_baseline_c"] = complete.month.map(monthly_normal)
    complete["temperature_month_anomaly_c"] = complete.temperature_mean_c - complete.temperature_month_baseline_c
    return d.reset_index(), complete.reset_index(), baseline.reset_index()

audit, weather, _ = aggregate_daily(hourly)
reference = weather.loc[weather.year.between(*BASELINE)].copy()
assert len(reference) == 5479 and reference.year.nunique() == 15
display(audit.loc[~audit.is_complete_day, ['service_date', 'observed_hours', 'expected_hours']])
display(weather[['service_date', 'temperature_mean_c', 'precipitation_total_mm',
                 'rain_total_mm', 'snowfall_total_cm', 'wind_max_kmh']].head())
print(f'Merge-ready: {len(weather):,} days and {weather.shape[1]} columns.')
print('Local coverage:', weather.service_date.min().date(), 'to', weather.service_date.max().date())
print('Day lengths:', weather.expected_hours.value_counts().sort_index().to_dict())
print(f'Seasonal reference: {len(reference):,} days in 15 complete calendar years, 2011–2025.')

## Define comparisons before plotting

Most comparative plots use only complete years **2011–2025**, avoiding unequal coverage
in partial 2010 and 2026. Event case studies can use the full complete-day export.
This 15-year sample is not an official 30-year climate normal.

For rain-versus-snow comparisons define rain as daily rain ≥1 mm and snow as daily new
snowfall ≥0.1 cm. Their four combinations form mutually exclusive categories, so a
100% stacked bar can sum correctly. **Neither threshold** includes trace amounts and
must not be labeled completely dry. A mixed day can contain rain and snow at different
hours; the category does not prove simultaneous precipitation.

In [ ]:
rain_flag = reference.rain_total_mm.ge(1)
snow_flag = reference.snowfall_total_cm.ge(0.1)
reference['weather_type'] = np.select(
    [rain_flag & snow_flag, rain_flag & ~snow_flag, ~rain_flag & snow_flag],
    ['Mixed', 'Rain only', 'Snow only'], default='Neither threshold',
)
TYPE_ORDER = ['Neither threshold', 'Rain only', 'Snow only', 'Mixed']
display(reference.weather_type.value_counts().reindex(TYPE_ORDER).rename('days').to_frame())
print(f'Rain ≥1 mm: {rain_flag.sum():,} days ({rain_flag.mean():.2%})')
print(f'Snow ≥0.1 cm: {snow_flag.sum():,} days ({snow_flag.mean():.2%})')
print('Mixed days are counted in both individual event rates, but once in the four-category chart.')

## 1. A time-series chart with pandas

**Question:** How did daily temperature vary through 2019?

**Library:** pandas plotting → Matplotlib axes

**How to build it:** Set the date as the index, select a Series, then call `.plot()`. Add a trailing 14-day rolling mean on the same axes. A trailing window uses current and past observations; the smoother is not a forecast.

In [ ]:
year2019 = weather.loc[weather.year.eq(2019)].set_index('service_date')
fig, ax = plt.subplots(figsize=(12, 4))
year2019.temperature_mean_c.plot(ax=ax, alpha=0.45, linewidth=0.9, label='Daily mean')
year2019.temperature_mean_c.rolling(14, min_periods=14).mean().plot(
    ax=ax, color='#c54f38', linewidth=2, label='Trailing 14-day mean',
)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set(title='Chicago daily temperature, 2019', xlabel='Local calendar date', ylabel='Temperature (°C)')
ax.legend()
show_figure(fig, '01_pandas_temperature_time_series.png')

**How to read it:** The thin line shows day-to-day variation and the thick line shows slower movement. Winter and summer are visibly different; smoothing hides some extremes. Change the window to 7 or 30 and compare how much detail disappears.

## 2. Histograms in a Matplotlib subplot grid

**Question:** What are the shapes of the weather distributions?

**Library:** Matplotlib

**How to build it:** `plt.subplots(2, 2)` creates four axes; `ax.hist(..., bins=...)` controls the bins. Include zero days in temperature/wind but show positive-only amount distributions with their zero shares explicitly reported.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
specifications = [
    ('temperature_mean_c', 'Daily mean temperature (°C)', False),
    ('rain_total_mm', 'Rain on positive-rain days (mm)', True),
    ('snowfall_total_cm', 'Snowfall on positive-snow days (cm)', True),
    ('wind_max_kmh', 'Daily maximum hourly wind (km/h)', False),
]
for ax, (column, label, positive_only) in zip(axes.flat, specifications):
    values = reference[column]
    zero_share = values.eq(0).mean()
    plotted = values[values.gt(0)] if positive_only else values
    ax.hist(plotted, bins=35, color='#1976a3', edgecolor='white')
    ax.set(xlabel=label, ylabel='Days')
    if positive_only:
        ax.set_title(f'Zero days omitted: {zero_share:.1%}; positive n={len(plotted):,}')
fig.suptitle('Distributions across complete years, 2011–2025')
fig.tight_layout()
show_figure(fig, '02_matplotlib_histograms.png')

**How to read it:** Rain and snowfall have many zeros and long right tails. A histogram limited to positive days describes event intensity, not the chance of an event. Counts change with bin width, so compare several bin settings before drawing conclusions.

## 3. Compare distributions fairly with Seaborn

**Question:** How different are winter and summer temperature distributions?

**Library:** Seaborn histplot

**How to build it:** Use `hue` for groups and `stat="density", common_norm=False` so each group has unit area. Common bins and translucent step shapes make the comparison readable without letting sample size determine the height.

In [ ]:
winter_summer = reference.loc[reference.season.isin(['Winter', 'Summer'])]
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(data=winter_summer, x='temperature_mean_c', hue='season',
             hue_order=['Winter', 'Summer'], bins=np.arange(-35, 41, 2),
             stat='density', common_norm=False, element='step', alpha=0.3, ax=ax)
ax.set(title='Winter versus summer daily mean temperature, 2011–2025',
       xlabel='Daily mean temperature (°C)', ylabel='Density (area = 1 per season)')
show_figure(fig, '03_seaborn_seasonal_histogram.png')

**How to read it:** The distributions differ in both location and spread. Density is not a count or a percentage at a single exact temperature; probabilities correspond to areas over temperature intervals.

## 4. Monthly box plots and extreme observations

**Question:** How do typical temperatures and variability change by month?

**Library:** Seaborn boxplot

**How to build it:** Pass the month column as `x` and daily temperature as `y`. Explicitly order months. The box spans the interquartile range, the line marks the median, and default whiskers extend to observations within 1.5 IQR.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=reference, x='month', y='temperature_mean_c', order=range(1, 13),
            color='#9ecae1', fliersize=2, linewidth=1, ax=ax)
ax.set_xticks(range(12), MONTHS)
ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
ax.set(title='Monthly distribution of daily mean temperature, 2011–2025',
       xlabel='Local month', ylabel='Daily mean temperature (°C)')
show_figure(fig, '04_monthly_temperature_boxplot.png')

**How to read it:** January and February have low medians and substantial variation. Dots outside the whiskers are unusual within that month; they are not automatically bad data. Whiskers are not confidence intervals.

## 5. Violin plots for positive rainfall

**Question:** Does rainfall intensity on rain-producing days differ by season?

**Library:** Seaborn violinplot

**How to build it:** Filter positive amounts, then use `cut=0` to prevent the estimated density extending outside observed values. `inner="quart"` marks quartiles. Equal-area normalization compares shape rather than the number of rainy days.

In [ ]:
positive_rain = reference.loc[reference.rain_total_mm.gt(0)]
fig, ax = plt.subplots(figsize=(10, 5))
sns.violinplot(data=positive_rain, x='season', y='rain_total_mm', order=SEASONS,
               cut=0, inner='quart', density_norm='area', color='#9ecae1', ax=ax)
ax.set(title='Rainfall intensity conditional on positive rain, 2011–2025',
       xlabel='Season', ylabel='Daily rain (mm; zero-rain days omitted)')
show_figure(fig, '05_positive_rain_violin.png')

**How to read it:** Wider parts indicate more common amounts within a season. The smooth shape depends on bandwidth and does not establish a physical multimodal process. This chart excludes zeros, so pair it with an occurrence-rate chart before claiming a season is rainier.

## 6. Grouped bars for rain-versus-snow frequency

**Question:** In each month, how often does rain or snow reach the chosen thresholds?

**Library:** pandas bar plotting

**How to build it:** Convert boolean flags to percentages with `groupby(...).mean()*100`. The resulting two-column DataFrame can draw grouped bars directly with `.plot.bar()`. Frequencies share a percentage unit, unlike rain and snow amounts.

In [ ]:
rates = reference.assign(rain_event=rain_flag, snow_event=snow_flag).groupby('month')[
    ['rain_event', 'snow_event']
].mean().mul(100)
rates.columns = ['Rain ≥1 mm', 'Snow ≥0.1 cm']
rates.index = MONTHS
ax = rates.plot.bar(figsize=(12, 5), color=['#1976a3', '#7851a9'], rot=0)
ax.set(title='Monthly rain and snow occurrence, 2011–2025', xlabel='Month', ylabel='Percent of days')
ax.yaxis.set_major_formatter(PercentFormatter(xmax=100))
show_figure(ax.figure, '06_grouped_rain_snow_frequency.png')
display(rates.round(1))

**How to read it:** The snow rate is concentrated in cold months. Rain and snow rates need not sum to 100% because mixed days satisfy both flags and many days satisfy neither. A different event threshold will change these rates.

## 7. Monthly amount bars with uncertainty intervals

**Question:** What are typical monthly rain and snow totals, and how much uncertainty is there across years?

**Library:** Matplotlib + NumPy bootstrap

**How to build it:** First sum within each complete year/month. Resample the 15 years as whole blocks with a fixed random seed; compute bootstrap monthly means and 95% percentile intervals. Draw separate rain and snow panels because their physical units differ.

In [ ]:
monthly_amounts = reference.groupby(['year', 'month'])[
    ['rain_total_mm', 'snowfall_total_cm']
].sum()
rng = np.random.default_rng(42)
resampled_years = rng.integers(0, 15, size=(4000, 15))
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
monthly_interval_tables = []
for ax, column, label, color in [
    (axes[0], 'rain_total_mm', 'Rain (mm/month)', '#1976a3'),
    (axes[1], 'snowfall_total_cm', 'Snowfall (cm/month)', '#7851a9'),
]:
    matrix = monthly_amounts[column].unstack('month').reindex(columns=range(1, 13))
    assert matrix.shape == (15, 12) and not matrix.isna().any().any()
    values = matrix.to_numpy()
    average = values.mean(axis=0)
    boot_means = values[resampled_years].mean(axis=1)
    low, high = np.quantile(boot_means, [0.025, 0.975], axis=0)
    errors = np.vstack([average - low, high - average])
    assert (errors >= -1e-12).all()
    ax.bar(range(1, 13), average, yerr=np.maximum(errors, 0), capsize=3, color=color)
    ax.set(ylabel=label)
    monthly_interval_tables.append(pd.DataFrame({
        'measure': column, 'month': range(1, 13), 'mean': average, 'ci_low': low, 'ci_high': high,
    }))
axes[0].set_title('Mean monthly amounts across 2011–2025; 95% year-block bootstrap intervals')
axes[1].set(xticks=range(1, 13), xticklabels=MONTHS, xlabel='Month')
fig.tight_layout()
show_figure(fig, '07_monthly_amount_bootstrap_bars.png')
write_csv(pd.concat(monthly_interval_tables, ignore_index=True), 'tutorial_monthly_amount_intervals.csv', REPORT)

**How to read it:** Bars estimate mean monthly amounts; intervals describe uncertainty in those historical means, not the spread of individual months or future forecasts. Resampling whole years retains within-year dependence, but assumes years are exchangeable and does not model climate trends or dependence between years.

## 8. A month-by-year heatmap

**Question:** Which years contain unusually warm or cold months within the sample?

**Library:** Seaborn heatmap + pandas pivot

**How to build it:** Aggregate monthly daily-mean temperatures, reshape with `unstack`, and pass the matrix to `heatmap`. A diverging palette centered on 0°C makes the freezing reference explicit.

In [ ]:
temperature_grid = reference.groupby(['year', 'month']).temperature_mean_c.mean().unstack('month')
fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(temperature_grid, cmap='RdBu_r', center=0, annot=True, fmt='.1f',
            linewidths=0.4, cbar_kws={'label': 'Mean daily temperature (°C)'}, ax=ax)
ax.set_xticklabels(MONTHS, rotation=0)
ax.set(title='Monthly mean temperature, complete years 2011–2025', xlabel='Month', ylabel='Year')
show_figure(fig, '08_month_year_temperature_heatmap.png')

**How to read it:** Compare values within the same month to distinguish interannual differences from the strong seasonal cycle. This chart is descriptive; it is not a trend significance test.

## 9. A density plot that preserves zero precipitation

**Question:** How is precipitation distributed across daily mean temperatures?

**Library:** Matplotlib hexbin

**How to build it:** Many days overlap at zero. Hexagonal binning shows density instead of plotting every overlapping point. Use `log1p(amount)` to retain zeros and compress the long tail; label ticks back in original millimeters. The color bar uses logarithmic counts.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
hb = ax.hexbin(reference.temperature_mean_c, np.log1p(reference.precipitation_total_mm),
               gridsize=40, mincnt=1, bins='log', cmap='viridis')
amount_ticks = np.array([0, 1, 5, 10, 25, 50, 150])
ax.set_yticks(np.log1p(amount_ticks), [str(x) for x in amount_ticks])
ax.set(title='Temperature and precipitation density, 2011–2025',
       xlabel='Daily mean temperature (°C)', ylabel='Daily precipitation (mm; log1p spacing)')
fig.colorbar(hb, ax=ax, label='Days per hexagon (log color scale)')
show_figure(fig, '09_temperature_precipitation_hexbin.png')

**How to read it:** The concentration near zero is real information and remains visible. Logarithmic spacing means equal vertical distances are not equal increases in millimeters. The pooled pattern also reflects seasonality, so it does not establish temperature as the cause of precipitation.

## 10. Small multiples: compare the same relationship by season

**Question:** Do rain-only, snow-only, and mixed days occupy different temperature ranges within each season?

**Library:** Seaborn relplot / FacetGrid

**How to build it:** `relplot` creates consistent axes across facets. Use `col` and `col_wrap` for panels, and one shared hue palette for weather types. Filter to days exceeding at least one event threshold so uninformative near-zero days do not dominate.

In [ ]:
event_days = reference.loc[reference.weather_type.ne('Neither threshold')]
grid = sns.relplot(data=event_days, x='temperature_mean_c', y='precipitation_total_mm',
                   hue='weather_type', hue_order=['Rain only', 'Snow only', 'Mixed'], palette=PALETTE,
                   col='season', col_order=SEASONS, col_wrap=2, kind='scatter',
                   alpha=0.5, s=22, height=3.1, aspect=1.35)
grid.set_axis_labels('Daily mean temperature (°C)', 'Daily precipitation (mm)')
grid.set_titles('{col_name}')
grid.figure.suptitle('Event-day precipitation by season, 2011–2025', y=1.04)
show_figure(grid.figure, '10_seasonal_event_scatter_facets.png')

**How to read it:** Fixed axes allow panel-to-panel comparison. A daily mean above 0°C can still occur on a snow day because snow can fall during a colder part of that day. Mixed here means rain and snow occurred during the same local day, not necessarily at the same hour.

## 11. Exceedance curves for extreme-event frequency

**Question:** Among positive-amount days, how often is an amount above a given value?

**Library:** Seaborn ecdfplot

**How to build it:** An empirical cumulative distribution avoids arbitrary histogram bins. `complementary=True` plots the share above each observed value. Show precipitation and snowfall separately and report the positive-day denominator.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, column, label, threshold, color in [
    (axes[0], 'precipitation_total_mm', 'Precipitation (mm)', 25, '#1976a3'),
    (axes[1], 'snowfall_total_cm', 'Snowfall (cm)', 10, '#7851a9'),
]:
    positive = reference.loc[reference[column].gt(0), column]
    sns.ecdfplot(x=positive, complementary=True, color=color, ax=ax)
    ax.axvline(threshold, color='black', linestyle='--', linewidth=1)
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
    ax.set(xlabel=label, ylabel='Share of positive-amount days above x',
           title=f'Positive n={len(positive):,}; reference threshold {threshold}')
fig.suptitle('Conditional exceedance distributions, 2011–2025')
fig.tight_layout()
show_figure(fig, '11_precipitation_snow_exceedance.png')

**How to read it:** Read a horizontal value from the curve as an event-size exceedance rate among positive days. The plotted complement uses values strictly above x; exported flags use inclusive thresholds (≥). These conditional rates must not be confused with the percentage of all days meeting a threshold.

## 12. A correlation heatmap and redundant predictors

**Question:** Which candidate weather features overlap strongly?

**Library:** Seaborn heatmap

**How to build it:** Compute Pearson correlations on selected continuous columns, mask duplicate triangles, and fix the color range from −1 to 1. A fixed range prevents small correlations looking misleadingly intense.

In [ ]:
correlation_columns = ['temperature_mean_c', 'temperature_min_c', 'temperature_max_c',
                       'precipitation_total_mm', 'rain_total_mm', 'snowfall_total_cm',
                       'wind_mean_kmh', 'wind_max_kmh']
labels = ['Mean temp', 'Min temp', 'Max temp', 'Precipitation', 'Rain', 'Snowfall', 'Mean wind', 'Max wind']
corr = reference[correlation_columns].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, mask=np.triu(np.ones_like(corr, dtype=bool), k=1),
            annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            xticklabels=labels, yticklabels=labels, square=True, ax=ax)
ax.set_title('Pearson correlations across complete years, 2011–2025')
plt.setp(ax.get_xticklabels(), rotation=40, ha='right')
show_figure(fig, '12_weather_correlation_heatmap.png')
write_csv(corr.rename_axis('variable').reset_index(), 'tutorial_reference_correlations.csv', REPORT)

**How to read it:** Mean/min/max temperature overlap, and total precipitation overlaps strongly with rain. Avoid including every correlated version in the final transit model. A weak Pearson correlation can still hide nonlinear associations, and these pooled correlations include seasonal variation.

## 13. A 100% stacked bar chart

**Question:** What fraction of each month belongs to mutually exclusive weather types?

**Library:** pandas stacked bar plotting

**How to build it:** `pd.crosstab(..., normalize="index")` normalizes within month; multiplying by 100 gives percentages. A stacked chart is valid because the four categories exhaust all days and do not overlap.

In [ ]:
type_shares = pd.crosstab(reference.month, reference.weather_type, normalize='index').mul(100)
type_shares = type_shares.reindex(index=range(1, 13), columns=TYPE_ORDER, fill_value=0)
assert np.allclose(type_shares.sum(axis=1), 100)
type_shares.index = MONTHS
ax = type_shares.plot.bar(stacked=True, figsize=(12, 5), rot=0,
                         color=[PALETTE[c] for c in TYPE_ORDER])
ax.set(title='Composition of local days by month, 2011–2025', xlabel='Month', ylabel='Share of days (%)', ylim=(0, 100))
ax.yaxis.set_major_formatter(PercentFormatter(xmax=100))
ax.legend(title='Threshold-based category', bbox_to_anchor=(1.01, 1), loc='upper left')
show_figure(ax.figure, '13_stacked_monthly_weather_types.png')

**How to read it:** Each complete bar totals 100%. This answers a different question from the overlapping rain/snow bars in lesson 6. Stacked segments away from the baseline are harder to compare precisely; use grouped bars when direct comparison is more important than composition.

## 14. Measure overlapping adverse-weather exposures

**Question:** When two adverse conditions occur, how often do they overlap?

**Library:** Seaborn heatmap + matrix operations

**How to build it:** Convert flags to an integer matrix and use `X.T @ X` for intersection counts. Divide by union counts to get Jaccard overlap. Unlike simple co-absence measures, this ignores the many days when neither condition occurs.

In [ ]:
flags = ['heavy_precipitation_day', 'heavy_snow_day', 'cold_day', 'hot_day', 'windy_day']
flag_labels = ['Precip ≥25 mm', 'Snow ≥10 cm', 'Min ≤−10°C', 'Max ≥30°C', 'Wind ≥40 km/h']
X = reference[flags].astype(int)
intersection = X.T @ X
counts = X.sum().to_numpy()
union = counts[:, None] + counts[None, :] - intersection.to_numpy()
jaccard = np.divide(intersection.to_numpy(), union, out=np.zeros_like(union, dtype=float), where=union > 0)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(jaccard * 100, annot=True, fmt='.1f', cmap='Blues', vmin=0, vmax=100,
            xticklabels=flag_labels, yticklabels=flag_labels, square=True,
            cbar_kws={'label': 'Intersection / union (%)'}, ax=ax)
ax.set_title('Adverse-condition overlap, 2011–2025 (diagonal = 100%)')
plt.setp(ax.get_xticklabels(), rotation=35, ha='right')
show_figure(fig, '14_adverse_condition_jaccard.png')
display(pd.DataFrame({'condition': flags, 'reference_days': counts}))

**How to read it:** Off-diagonal values compare overlap relative to days with either event. These are not conditional probabilities or effects on transit. Rare conditions can produce unstable percentages; inspect the counts and do not infer relationships from color alone.

## 15. An annotated multi-panel event study

**Question:** What weather preceded and accompanied the January 2019 cold spell?

**Library:** Matplotlib shared axes, fill_between, annotations

**How to build it:** Create vertically aligned axes with a shared date axis. Use a temperature band for daily sampled min/max, a line for the mean, snowfall bars, and a separate wind panel. Shade the severe-cold interval and annotate the minimum.

In [ ]:
case = weather.loc[weather.service_date.between('2019-01-25', '2019-02-04')]
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True,
                         gridspec_kw={'height_ratios': [2, 1, 1]})
axes[0].fill_between(case.service_date, case.temperature_min_c, case.temperature_max_c,
                     alpha=0.2, color='#1976a3', label='Hourly sampled min–max')
axes[0].plot(case.service_date, case.temperature_mean_c, marker='o', label='Daily mean')
axes[0].axhline(-20, color='#c54f38', linestyle='--', label='−20°C reference')
axes[0].annotate('−32.5°C minimum', xy=(pd.Timestamp('2019-01-31'), -32.5),
                 xytext=(pd.Timestamp('2019-02-01'), -29), arrowprops={'arrowstyle': '->'})
axes[0].set(ylabel='Temperature (°C)', title='January 2019 cold spell: weather context only')
axes[0].legend(fontsize=8, loc='upper left')
axes[1].bar(case.service_date, case.snowfall_total_cm, color='#7851a9')
axes[1].set(ylabel='New snowfall (cm)')
axes[2].plot(case.service_date, case.wind_max_kmh, marker='o', color='#b67524')
axes[2].set(ylabel='Max hourly wind (km/h)', xlabel='Chicago local calendar date')
for ax in axes:
    ax.axvspan(pd.Timestamp('2019-01-29'), pd.Timestamp('2019-02-01'), color='#c54f38', alpha=0.07)
fig.autofmt_xdate()
fig.tight_layout()
show_figure(fig, '15_annotated_january2019_event.png')
display(case[['service_date', 'temperature_min_c', 'temperature_mean_c',
              'snowfall_total_cm', 'snowfall_total_cm_lag1', 'severe_cold_day']].round(2))

**How to read it:** The shaded interval covers January 29–31. January 28 snowfall preceded the deepest cold, supporting examination of lagged snowfall after the future transit merge. This chart does not estimate ridership loss, delays, wind chill, or remaining snow on the ground.

## 16. A calendar heatmap from daily observations

**Question:** When did extreme cold days cluster during 2019?

**Library:** pandas pivot + Seaborn heatmap

**How to build it:** Anchor week zero to a Monday, calculate week offsets, and pivot weekday versus week. This avoids ISO-week year-boundary errors. Gray cells represent dates outside 2019, not missing weather records.

In [ ]:
calendar_days = weather.loc[weather.year.eq(2019), ['service_date', 'temperature_min_c']].copy()
start_of_year = pd.Timestamp('2019-01-01')
monday_anchor = start_of_year - pd.Timedelta(days=start_of_year.weekday())
calendar_days['weekday_index'] = calendar_days.service_date.dt.dayofweek
calendar_days['week_index'] = ((calendar_days.service_date - monday_anchor).dt.days // 7).astype(int)
calendar_grid = calendar_days.pivot(index='weekday_index', columns='week_index', values='temperature_min_c').reindex(range(7))
cmap = plt.get_cmap('RdBu_r').copy()
cmap.set_bad('#e4e8eb')
fig, ax = plt.subplots(figsize=(15, 3.5))
ax.set_facecolor('#e4e8eb')
sns.heatmap(calendar_grid, cmap=cmap, center=0, linewidths=0.35,
            yticklabels=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'],
            cbar_kws={'label': 'Daily minimum (°C)'}, ax=ax)
week_ticks = np.arange(0, calendar_grid.shape[1], 4)
ax.set_xticks(week_ticks + 0.5,
              [(monday_anchor + pd.Timedelta(weeks=int(w))).strftime('%b %d') for w in week_ticks], rotation=40)
ax.set(title='2019 calendar of daily minimum temperature; gray = outside the year',
       xlabel='Week beginning Monday', ylabel='Weekday')
show_figure(fig, '16_calendar_minimum_temperature.png')

**How to read it:** Vertical clusters show consecutive weather exposure across weekdays. This is a calendar view of temperature, not evidence of weekday effects on weather or ridership.

## 17. An hourly seasonal heatmap

**Question:** How does the typical daily temperature cycle vary by month?

**Library:** Seaborn heatmap on hourly data

**How to build it:** Return to complete-day hourly records, keep the 2011–2025 reference, then pivot month versus local hour using mean temperature. A plot of daily aggregates alone cannot answer a clock-hour question.

In [ ]:
hourly_reference = hourly.loc[hourly.service_date.isin(reference.service_date)]
hourly_cycle = hourly_reference.assign(month=hourly_reference.time_local.dt.month).pivot_table(
    index='month', columns='local_hour', values='temperature_c', aggfunc='mean',
)
fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(hourly_cycle, cmap='RdBu_r', center=0, yticklabels=MONTHS,
            cbar_kws={'label': 'Mean hourly temperature (°C)'}, ax=ax)
ax.set(title='Mean temperature by Chicago clock hour and month, 2011–2025',
       xlabel='Local hour (0–23)', ylabel='Month')
show_figure(fig, '17_hourly_monthly_temperature_cycle.png')

**How to read it:** Within a month, colors show the average daily temperature cycle. These averages hide extreme hours. The repeated fall-back clock hour contributes both valid observations; no nonexistent spring-forward hour is invented. Hourly weather cannot reveal hourly transit effects from daily boarding totals.

## 18. Paired AM–PM comparison and a difference histogram

**Question:** On the same weekdays, how do chosen AM and PM temperature windows differ?

**Library:** Seaborn scatterplot + pandas histogram

**How to build it:** Use one row per weekday to preserve pairing. A scatterplot with an identity line shows which window is warmer; a histogram of PM minus AM shows the difference distribution. Pairing avoids confusing day-to-day variation with time-of-day differences.

In [ ]:
paired = reference.loc[~reference.is_weekend].copy()
paired['pm_minus_am_c'] = paired.pm_peak_temperature_mean_c - paired.am_peak_temperature_mean_c
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=paired, x='am_peak_temperature_mean_c', y='pm_peak_temperature_mean_c',
                hue='season', hue_order=SEASONS, alpha=0.35, s=18, ax=axes[0])
limits = [paired[['am_peak_temperature_mean_c', 'pm_peak_temperature_mean_c']].min().min(),
          paired[['am_peak_temperature_mean_c', 'pm_peak_temperature_mean_c']].max().max()]
axes[0].plot(limits, limits, 'k--', linewidth=1, label='AM = PM')
axes[0].set(xlabel='AM mean temperature (°C)', ylabel='PM mean temperature (°C)', title='Paired weekday windows')
paired.pm_minus_am_c.plot.hist(ax=axes[1], bins=35, color='#1976a3', edgecolor='white')
axes[1].axvline(0, color='black', linestyle='--')
axes[1].set(xlabel='PM minus AM mean temperature (°C)', ylabel='Weekdays', title='Same-day difference distribution')
fig.suptitle('07–09 versus 16–18 clock-hour labels; 2011–2025, holidays included')
fig.tight_layout()
show_figure(fig, '18_paired_peak_temperature_comparison.png')

**How to read it:** Points above the diagonal and positive differences indicate a warmer PM window. These are chosen weather windows, not official CTA peak definitions. No conclusion about AM versus PM ridership is possible without transit data at that resolution.

## 19. Consecutive-event durations

**Question:** How persistent are wet and snow days?

**Library:** pandas run grouping + bar plotting

**How to build it:** Detect changes in a boolean event flag with `ne(shift())`, use `cumsum()` to label runs, then count the lengths of true runs. Event definitions must be explicit: wet means precipitation ≥1 mm/day and snow means new snowfall ≥0.1 cm/day.

In [ ]:
run_lengths = {}
for column, label in [('wet_day', 'Wet ≥1 mm'), ('snow_day', 'Snow ≥0.1 cm')]:
    flag = reference[column]
    groups = flag.ne(flag.shift()).cumsum()
    run_lengths[label] = reference.loc[flag].groupby(groups.loc[flag]).size()
duration_counts = pd.DataFrame({label: lengths.value_counts() for label, lengths in run_lengths.items()}).fillna(0)
duration_counts = duration_counts.reindex(range(1, int(duration_counts.index.max()) + 1), fill_value=0)
ax = duration_counts.plot.bar(figsize=(12, 4), color=['#1976a3', '#7851a9'], rot=0)
ax.set(title='Observed consecutive-event runs within 2011–2025',
       xlabel='Consecutive local days in a run', ylabel='Number of runs')
show_figure(ax.figure, '19_weather_run_lengths.png')
display(pd.DataFrame({label: lengths.describe() for label, lengths in run_lengths.items()}).round(2))

**How to read it:** Most runs are short, while longer episodes motivate lagged weather and serial-correlation-aware analysis. Runs touching the reference start/end are boundary-censored and may continue outside the window. Snow-day runs indicate consecutive new snowfall, not persistent snow cover.

## Save reproducible weather outputs and verify the result

The daily CSV retains the original 51-column merge schema. Additional tutorial-specific
tables and 19 PNG figures are saved alongside it when the notebook runs. The complete
feature dictionary documents units, thresholds, lag windows, and intentional missingness.
The helper functions below produce summaries; they are included here so this notebook
remains self-contained.

In [ ]:
def summaries(d, baseline):
    correlation = d[["temperature_mean_c", "precipitation_total_mm", "rain_total_mm", "snowfall_total_cm", "wind_max_kmh"]].corr()
    write_csv(correlation.rename_axis("variable").reset_index(), "weather_correlations.csv", REPORT)
    flags = [c for c in d if c.endswith("_day") and c != "is_complete_day"]
    monthly = d.assign(period=d.service_date.dt.to_period("M")).groupby("period").agg(
        complete_days=("service_date", "size"), temperature_mean_c=("temperature_mean_c", "mean"),
        temperature_min_c=("temperature_min_c", "min"), temperature_max_c=("temperature_max_c", "max"),
        precipitation_total_mm=("precipitation_total_mm", "sum"), snowfall_total_cm=("snowfall_total_cm", "sum"),
        wind_mean_kmh=("wind_mean_kmh", "mean"))
    monthly["expected_days"] = monthly.index.days_in_month
    monthly["is_complete_month"] = monthly.complete_days.eq(monthly.expected_days)
    monthly.index = monthly.index.astype(str)
    write_csv(monthly.reset_index(), "monthly_weather.csv")
    annual = d.groupby("year").agg(complete_days=("service_date", "size"),
        temperature_mean_c=("temperature_mean_c", "mean"), precipitation_total_mm=("precipitation_total_mm", "sum"),
        snowfall_total_cm=("snowfall_total_cm", "sum"))
    annual["expected_days"] = [366 if pd.Timestamp(y, 12, 31).is_leap_year else 365 for y in annual.index]
    annual["is_complete_year"] = annual.complete_days.eq(annual.expected_days)
    write_csv(annual.reset_index(), "annual_weather.csv")
    seasonal = baseline.groupby("season").agg(days=("service_date", "size"),
        temperature_mean_c=("temperature_mean_c", "mean"), precipitation_mean_daily_mm=("precipitation_total_mm", "mean"),
        snowfall_mean_daily_cm=("snowfall_total_cm", "mean"), wind_mean_kmh=("wind_mean_kmh", "mean"))
    for flag in flags:
        seasonal[flag + "_pct"] = baseline.groupby("season")[flag].mean() * 100
    write_csv(seasonal.reset_index(), "seasonal_summary.csv", REPORT)
    climatology = baseline.groupby("month").agg(days=("service_date", "size"),
        temperature_mean_c=("temperature_mean_c", "mean"), temperature_p10_c=("temperature_mean_c", lambda x: x.quantile(.1)),
        temperature_p90_c=("temperature_mean_c", lambda x: x.quantile(.9)),
        precipitation_mean_daily_mm=("precipitation_total_mm", "mean"),
        snowfall_mean_daily_cm=("snowfall_total_cm", "mean"))
    write_csv(climatology.reset_index(), "monthly_baseline_2011_2025.csv", REPORT)
    frequency = pd.DataFrame({"condition": flags + ["any_adverse_condition"],
        "days": [int(d[c].sum()) for c in flags + ["any_adverse_condition"]],
        "percent_complete_days": [100*d[c].mean() for c in flags + ["any_adverse_condition"]]})
    write_csv(frequency, "condition_frequency.csv", REPORT)
    sensitivity = []
    for variable, unit, thresholds, direction in [
        ("precipitation_total_mm", "mm", [10, 20, 25, 50], "ge"),
        ("snowfall_total_cm", "cm", [1, 5, 10, 15], "ge"),
        ("temperature_min_c", "C", [-10, -15, -20], "le"),
        ("temperature_max_c", "C", [30, 32, 35], "ge"),
        ("wind_max_kmh", "km/h", [30, 40, 50], "ge")]:
        for threshold in thresholds:
            mask = d[variable].ge(threshold) if direction == "ge" else d[variable].le(threshold)
            sensitivity.append(dict(variable=variable, unit=unit, comparison=direction,
                threshold=threshold, days=int(mask.sum()), percent_complete_days=100*mask.mean()))
    write_csv(pd.DataFrame(sensitivity), "threshold_sensitivity.csv", REPORT)
    events = []
    for variable, ascending in [("temperature_min_c", True), ("temperature_max_c", False),
            ("precipitation_total_mm", False), ("snowfall_total_cm", False), ("wind_max_kmh", False)]:
        top = d.sort_values([variable, "service_date"], ascending=[ascending, True]).head(10).copy()
        top.insert(0, "rank", range(1, len(top)+1))
        top.insert(0, "ranked_by", variable)
        events.append(top)
    write_csv(pd.concat(events, ignore_index=True), "top_weather_days.csv", REPORT)
    cold = d.loc[d.severe_cold_day, ["service_date", "temperature_min_c", "temperature_mean_c"]].copy()
    if not cold.empty:
        cold["spell"] = cold.service_date.diff().dt.days.ne(1).cumsum()
        spells = cold.groupby("spell").agg(start_date=("service_date", "min"), end_date=("service_date", "max"),
            days=("service_date", "size"), minimum_c=("temperature_min_c", "min"))
        write_csv(spells.reset_index(drop=True), "severe_cold_spells.csv", REPORT)
    write_csv(d.loc[d.service_date.between("2019-01-27", "2019-02-02")], "january_2019_case_study.csv", REPORT)
    write_csv(d.loc[d.year.between(2017, 2019)], "daily_weather_2017_2019.csv")
    return monthly, annual, seasonal, climatology, frequency

def dictionary(d):
    definitions = {
        "service_date": ("YYYY-MM-DD", "Chicago local calendar date; unique merge key, timezone removed only after local conversion."),
        "observed_hours": ("hours", "Count of available hourly records for the local day."),
        "expected_hours": ("hours", "Local-day length: 23, 24, or 25 hours according to America/Chicago DST."),
        "is_complete_day": ("boolean", "Observed hours equal expected hours; all rows in merge-ready file are complete."),
        "year": ("year", "Year of local date."), "month": ("1–12", "Month of local date."),
        "weekday": ("name", "Weekday of local date."), "is_weekend": ("boolean", "Saturday/Sunday, not a holiday indicator."),
        "season": ("category", "Meteorological season: DJF, MAM, JJA, SON; year is calendar year."),
        "wet_hours": ("hours", "Hours with precipitation >=0.1 mm."),
        "snow_hours": ("hours", "Hours with snowfall >0 cm."),
        "freezing_hours": ("hours", "Hours with temperature <=0°C; not proof of surface ice."),
        "cold_hours_le_minus10c": ("hours", "Hours with temperature <=−10°C."),
        "hot_hours_ge_30c": ("hours", "Hours with temperature >=30°C."),
        "windy_hours_ge_40kmh": ("hours", "Hours with 10m sustained wind >=40 km/h; not gusts."),
        "wet_day": ("boolean", "Daily precipitation >=1 mm."),
        "heavy_precipitation_day": ("boolean", "Daily precipitation >=25 mm (liquid-water equivalent; includes snow)."),
        "snow_day": ("boolean", "Daily snowfall >=0.1 cm."),
        "heavy_snow_day": ("boolean", "Daily snowfall >=10 cm; new snowfall, not snow depth."),
        "cold_day": ("boolean", "Daily hourly minimum <=−10°C."),
        "severe_cold_day": ("boolean", "Daily hourly minimum <=−20°C."),
        "hot_day": ("boolean", "Daily hourly maximum >=30°C."),
        "very_hot_day": ("boolean", "Daily hourly maximum >=35°C."),
        "windy_day": ("boolean", "Daily maximum hourly 10m wind >=40 km/h."),
        "freeze_thaw_day": ("boolean", "Daily min <0°C and max >0°C; potential freeze/thaw exposure, not observed icing."),
        "adverse_condition_count": ("count", "Sum of heavy precipitation, heavy snow, cold, hot, and windy flags; overlaps allowed."),
        "any_adverse_condition": ("boolean", "At least one of the five adverse conditions; exploratory, not an official alert."),
        "temperature_month_baseline_c": ("°C", "Mean daily temperature for this calendar month using complete years 2011–2025; descriptive in-sample reference."),
        "temperature_month_anomaly_c": ("°C", "Daily mean minus monthly 2011–2025 reference; not a day-of-year anomaly or official climate normal."),
    }
    for col in d:
        if col in definitions: continue
        if col.endswith("_lag1"):
            base = col.removesuffix("_lag1"); unit = "cm" if "snowfall" in base else "°C" if "temperature" in base else "mm"
            definitions[col] = (unit, f"Previous calendar day's {base}; first row missing by design.")
        elif col.endswith("_prior3d"):
            definitions[col] = ("cm" if "snowfall" in col else "mm", "Sum of the preceding three complete calendar days, excluding today; first three rows missing.")
        elif col.startswith(("am_peak_", "pm_peak_")):
            unit = "°C" if "temperature" in col else "km/h" if "wind" in col else "cm" if "snowfall" in col else "mm"
            window = "07:00–09:59" if col.startswith("am") else "16:00–18:59"
            operation = "mean" if "temperature" in col else "maximum" if "wind" in col else "sum"
            definitions[col] = (unit, f"{operation.title()} during {window} Chicago time, Monday–Friday including holidays. Missing on weekends by design.")
        else:
            unit = "°C" if "temperature" in col else "km/h" if "wind" in col else "cm" if "snowfall" in col else "mm"
            operation = "mean" if "mean" in col else "minimum" if "min" in col else "maximum" if "max" in col else "sum"
            definitions[col] = (unit, f"Local-day {operation} of hourly {col.split('_')[0]}; precipitation includes rain and snow water equivalent, snowfall is new snowfall depth." if "precipitation" in col else f"Local-day {operation} of hourly {col.split('_')[0]}; extrema refer to sampled hourly values.")
    frame = pd.DataFrame([dict(column=c, dtype=str(d[c].dtype), units=definitions[c][0], definition=definitions[c][1]) for c in d])
    write_csv(frame, "data_dictionary.csv", REPORT)

write_csv(weather, 'daily_weather_chicago.csv')
write_csv(audit, 'daily_coverage_audit.csv')
write_csv(hourly, 'hourly_weather_chicago.csv')
monthly, annual, seasonal, climatology, frequency = summaries(weather, reference)
dictionary(weather)
quality = {
    'source_file': str(SOURCE.relative_to(ROOT)), 'source_sha256': source_checksum,
    'metadata': source_metadata, 'source_timezone': 'UTC', 'analysis_timezone': TIMEZONE,
    'hourly_rows': len(hourly), 'utc_start': str(hourly.time_utc.min()), 'utc_end': str(hourly.time_utc.max()),
    'complete_local_days': len(weather), 'merge_start': str(weather.service_date.min().date()),
    'merge_end': str(weather.service_date.max().date()), 'daily_feature_columns': weather.shape[1],
    'incomplete_local_days': json.loads(audit.loc[~audit.is_complete_day,
        ['service_date', 'observed_hours', 'expected_hours']].to_json(orient='records', date_format='iso')),
    'baseline_years': list(BASELINE), 'baseline_days': len(reference),
    'day_lengths': {str(k): int(v) for k, v in weather.expected_hours.value_counts().sort_index().items()},
    'full_years': list(range(2011, 2026)),
    'python_version': platform.python_version(), 'pandas_version': pd.__version__,
    'numpy_version': np.__version__, 'matplotlib_version': matplotlib.__version__, 'seaborn_version': sns.__version__,
    'missing_measurements': 0, 'duplicate_utc_timestamps': 0, 'missing_hourly_timestamps': 0,
    'tutorial_figures': 19,
}
(REPORT / 'quality_report.json').write_text(json.dumps(quality, indent=2) + '\n')
display(seasonal.reset_index()[['season', 'days', 'temperature_mean_c', 'precipitation_mean_daily_mm', 'snowfall_mean_daily_cm']].round(2))
display(frequency.round(2))
print('Daily CSV:', (DATA / 'daily_weather_chicago.csv').relative_to(ROOT))
print('Dictionary:', (REPORT / 'data_dictionary.csv').relative_to(ROOT))
print('Charts:', FIGURES.relative_to(ROOT))

## Findings to carry into the later transit analysis

- The source has 146,448 continuous hourly records. Time-zone conversion yields **6,101
  complete local days**, January 9, 2010–September 22, 2026, plus two incomplete boundary dates.
- In the complete-year reference, **1,656 days have rain ≥1 mm**, **588 have snowfall
  ≥0.1 cm**, and **132 meet both thresholds**. These event definitions matter.
- Seasonality is strong: mean winter temperature is about **−1.99°C**, compared with
  **22.37°C in summer**. Weather comparisons across all months require seasonal controls.
- The full-period minimum sampled temperature is **−32.5°C on January 31, 2019**.
  The three-day severe-cold spell follows snowfall on January 28. Same-day and prior-day
  weather are plausible candidate exposures for the later joined analysis.
- Rain and total precipitation overlap strongly. Avoid adding them together or treating
  every temperature/precipitation summary as an independent predictor.
- Rare events, serial dependence, changing ridership, and the COVID disruption need explicit
  treatment. These plots describe weather; they do not establish transit impacts.

## How to merge bus, rail, and weather later

Your CTA file already has **bus and rail totals on the same daily row**. Use a single
date merge; separate bus and rail merges are unnecessary. The example below is deliberately
shown as text and is not executed by this weather notebook.

1. Parse both date columns explicitly and check whether the CTA date is a calendar day or
   an operational service day. Establish any overnight cutoff before claiming exact alignment.
2. Keep a left join so unmatched transportation dates remain visible. Dates outside weather
   coverage are expected to be unmatched; missing weather does not mean dry or mild weather.
3. Check uniqueness and use `validate='one_to_one'` for daily totals. For station/route tables
   with repeated dates, use `many_to_one` instead.
4. Start with a small, interpretable weather subset. Retain CTA's existing day-type field;
   weather's weekend flag does not replace holiday/service classifications.

```python
transport = pd.read_csv(
    'datasets/CTA_-_Ridership_-_Daily_Boarding_Totals_20260924.csv',
    thousands=',',
)
transport['service_date'] = pd.to_datetime(
    transport['service_date'], format='%m/%d/%Y', errors='raise'
).dt.normalize()
wx = pd.read_csv(
    'datasets/processed/weather/daily_weather_chicago.csv',
    parse_dates=['service_date'],
)
assert transport.service_date.is_unique
assert wx.service_date.is_unique
weather_columns = [
    'service_date', 'temperature_mean_c', 'temperature_min_c',
    'precipitation_total_mm', 'snowfall_total_cm', 'wind_max_kmh',
    'cold_day', 'hot_day', 'snow_day', 'heavy_precipitation_day',
    'snowfall_total_cm_lag1', 'snowfall_total_cm_prior3d',
]
combined = transport.merge(
    wx[weather_columns], on='service_date', how='left',
    validate='one_to_one', indicator='weather_match',
)
assert len(combined) == len(transport)
print(combined.weather_match.value_counts())
unmatched = combined.loc[combined.weather_match.ne('both'), 'service_date']
print(unmatched.min(), unmatched.max())

# Begin with the same pre-COVID period already used in your transportation analysis.
study = combined.loc[
    combined.service_date.between('2017-01-01', '2019-12-31')
    & combined.weather_match.eq('both')
].copy()
long_ridership = study.melt(
    id_vars=['service_date', 'day_type', 'temperature_mean_c',
             'precipitation_total_mm', 'snowfall_total_cm'],
    value_vars=['bus', 'rail_boardings'],
    var_name='mode', value_name='boardings',
)
```

The long format gives one date–mode row and is convenient for `sns.relplot(..., col='mode')`
or grouped bars. Compare bus and rail **relative to their own calendar-matched baselines**,
because their boarding levels differ. Adjust for weekday, holidays, month/season, year or
long-term trend, and COVID when using the full record. Explore nonlinear temperature
effects and lagged exposures, and account for serial correlation. Fit scaling/reference
values only on training data if you build a predictive model. Daily boardings alone
cannot establish hourly effects or service reliability; those need time-resolved transit
or actual service/delay data.

## Practice exercises

1. Change histogram bins from 35 to 15 and then 60. Which visual features persist?
2. Change the rain-event threshold from 1 mm to 5 mm in a copy of the chart preparation.
   Recalculate the bars and weather types consistently. Do not silently change the exported schema.
3. Rebuild the monthly boxplot for maximum wind. Use the correct km/h axis label.
4. Modify the calendar heatmap to show snowfall, using a sequential color palette.
5. Compare precipitation and snowfall exceedance rates using all-day denominators instead
   of positive-day denominators. Explain why the percentages become smaller.

## Limits

The export represents one grid location and lacks exact model/download provenance and
independent station validation. It does not include gusts, humidity, visibility, wind
chill, freezing rain, road conditions, snow depth, official alerts, or transit delays.
Freeze/thaw exposure is not observed icing. Averages can hide localized storms and
short extreme episodes. Thresholds are exploratory and are not official warning criteria.
Bootstrap intervals and pooled correlations do not remove confounding or prove causality.

## Final independent checks

Verify date uniqueness, DST examples, source integrity, conserved precipitation/rain/snow
totals, intended missingness, and an unchanged merge-ready schema. The following checks
do not read or modify the transportation files.

In [ ]:
assert weather.service_date.is_unique and weather.is_complete_day.all()
assert weather.service_date.equals(audit.loc[audit.is_complete_day, 'service_date'].reset_index(drop=True))
assert len(weather) == 6101 and weather.shape[1] == 51
assert len(reference) == 5479
assert hashlib.sha256(SOURCE.read_bytes()).hexdigest() == source_checksum
indexed = weather.set_index('service_date')
assert indexed.loc['2019-03-10', 'observed_hours'] == 23
assert indexed.loc['2019-11-03', 'observed_hours'] == 25
assert indexed.loc['2019-01-31', 'temperature_min_c'] == -32.5
for hourly_column, daily_column in [
    ('precipitation_mm', 'precipitation_total_mm'), ('rain_mm', 'rain_total_mm'),
    ('snowfall_cm', 'snowfall_total_cm'),
]:
    np.testing.assert_allclose(hourly[hourly_column].sum(), audit[daily_column].sum(), atol=1e-6)
    daily_expected = hourly.groupby('service_date')[hourly_column].sum().reindex(indexed.index)
    np.testing.assert_allclose(indexed[daily_column], daily_expected, atol=1e-6)
expected_nulls = {
    c: int(weather.is_weekend.sum()) if c.startswith(('am_peak_', 'pm_peak_'))
    else 1 if c.endswith('_lag1') else 3 if c.endswith('_prior3d') else 0
    for c in weather.columns
}
assert weather.isna().sum().to_dict() == expected_nulls
assert len(list(FIGURES.glob('[0-9][0-9]_*.png'))) == 19
assert np.allclose(type_shares.sum(axis=1), 100)
assert len(reference.weather_type) == reference.weather_type.value_counts().sum()
print('PASS: all checks. Weather export retains 6,101 rows and 51 columns.')
print('19 chart lessons completed; transportation data was not read or modified.')